# Master Workflow - Grain-Size Analysis (Google Colab)

Runs **all seven analyses** on a single grain-size CSV and generates every output:

1. Histogram (grain-size frequency)
2. Frequency distribution curve (modality)
3. Cumulative curve + Folk & Ward statistics
4. Passega C-M (transport mechanism)
5. Folk (1954) ternary classification
6. Shepard (1954) ternary classification
7. Visher (1969) log-probability transport-population analysis

**Outputs:** per-sample figures (PNG 300dpi / TIFF / SVG / PDF), summary CSVs,
and professional PDF + HTML reports.

## Step 1 - Setup

In [ ]:
# === Google Colab setup ===================================================
# Installs dependencies and fetches the grainsize package + sample data.
import sys, subprocess, os

def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=False)

# 1. Dependencies (all public PyPI packages)
sh(f"{sys.executable} -m pip install -q numpy pandas matplotlib scipy openpyxl reportlab")

# 2. Get the package. If you cloned the repo to Drive, point PROJECT_DIR there.
#    Otherwise this cell expects the `grainsize/` folder next to the notebook
#    (upload the repo folder, or clone from GitHub).
PROJECT_DIR = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_DIR, "grainsize")):
    # Try one level up (if notebook is inside notebooks/)
    up = os.path.dirname(PROJECT_DIR)
    if os.path.isdir(os.path.join(up, "grainsize")):
        PROJECT_DIR = up
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)   # so sample_data/ and output/ resolve correctly
print("PROJECT_DIR =", PROJECT_DIR)

# 3. (Optional) mount Google Drive to save outputs there
try:
    from google.colab import drive
    MOUNT = False  # set True to mount
    if MOUNT:
        drive.mount("/content/drive")
except Exception:
    pass

import grainsize as gs
print("grainsize version:", gs.__version__)

## Step 2 - Choose input

In [ ]:
# === Choose your input CSV ================================================
# Option A - use a bundled example dataset:
CSV_PATH = "sample_data/river.csv"    # <-- change to any file in sample_data/

# Option B (Colab) - upload your own CSV instead:
#   from google.colab import files
#   up = files.upload()
#   CSV_PATH = list(up.keys())[0]

# Settings for YOUR data (defaults suit the bundled examples):
SIZE_UNIT  = "phi"         # "phi", "mm", or "um"
VALUE_TYPE = "frequency"   # "frequency" or "cumulative"
OUTDIR     = "output"

## Step 3 - Run the full workflow

In [ ]:
out = gs.run_workflow(
    CSV_PATH,
    outdir=OUTDIR,
    size_unit=SIZE_UNIT,
    value_type=VALUE_TYPE,
    figure_formats=("png", "tiff", "svg", "pdf"),
    make_reports=True,
)

print("\nSamples analysed:", out["meta"]["n_samples"])
print("Files written    :", len(out["paths"]))

## Step 4 - Inspect the summary table

In [ ]:
out["summary_df"]

## Step 5 - View the dataset-level diagrams inline

In [ ]:
from IPython.display import Image, display
for kind in ("passega_cm", "folk_ternary", "shepard_ternary"):
    p = out["dataset_figures"].get(kind)
    if p:
        print(kind)
        display(Image(filename=p))

## Step 6 - Open the reports
The PDF and HTML reports are in `output/reports/` and `output/html/`.

In [ ]:
import os
dataset = out["meta"]["dataset"]
print("PDF :", os.path.join(OUTDIR, "reports", dataset + "_report.pdf"))
print("HTML:", os.path.join(OUTDIR, "html", dataset + "_report.html"))
print("CSV :", os.path.join(OUTDIR, "csv", dataset + "_summary.csv"))

# In Colab you can download them:
# from google.colab import files
# files.download(os.path.join(OUTDIR, "reports", dataset + "_report.pdf"))

## Batch mode - process every dataset in `sample_data/`

Uncomment to run the whole example library at once.

In [ ]:
# import glob
# for f in sorted(glob.glob("sample_data/*.csv")):
#     gs.run_workflow(f, outdir=OUTDIR, figure_formats=("png",), make_reports=True)